# 03 模型训练与 MLflow 实验记录

**目标**：训练三个候选模型，把参数、指标、混淆矩阵全部记录到 MLflow，方便横向对比。

注意数据特点：流失样本只占约 17%，**准确率会骗人**——全预测「不流失」也有 83% 准确率。
所以用 `class_weight='balanced'` 并对齐关注 F1 与召回率。

In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet('../../data/processed/clean.parquet')
prep = joblib.load('../../models/preprocessor.pkl')

NUM_COLS = ['age', 'tenure_months', 'monthly_charge', 'total_charge',
            'support_calls', 'is_active', 'avg_monthly', 'support_per_year']
CAT_COLS = ['contract_type', 'payment_method']

X = prep.transform(df[NUM_COLS + CAT_COLS])
y = df['churn'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'训练 {X_train.shape[0]} 条 / 测试 {X_test.shape[0]} 条')

训练 1200 条 / 测试 300 条


## 1. 基线：一个不懂业务的模型

先看看「全猜不流失」能拿多少准确率。

In [7]:
from sklearn.metrics import accuracy_score, f1_score

naive = (y_test * 0 + 0)
print(f'全猜不流失: accuracy={accuracy_score(y_test, naive):.4f}  f1={f1_score(y_test, naive, zero_division=0):.4f}')

全猜不流失: accuracy=0.8267  f1=0.0000


## 2. 接入 MLflow

In [8]:
import mlflow

mlflow.set_tracking_uri('sqlite:///../../mlflow.db')
mlflow.set_experiment('churn-prediction')
print('当前实验:', mlflow.get_experiment_by_name('churn-prediction').name)

当前实验: churn-prediction


## 3. 训练三个候选模型

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, precision_score,
                             recall_score, roc_auc_score)
import matplotlib.pyplot as plt

candidates = [
    ('LogReg-C0.1', LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced'),
     {'model': 'LogisticRegression', 'C': 0.1}),
    ('LogReg-C1.0', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced'),
     {'model': 'LogisticRegression', 'C': 1.0}),
    ('RandomForest-200', RandomForestClassifier(n_estimators=200, max_depth=8,
                                                class_weight='balanced', random_state=42),
     {'model': 'RandomForest', 'n_estimators': 200, 'max_depth': 8}),
]

results = []
for name, model, params in candidates:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        m = {
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'recall': recall_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, proba),
        }
        mlflow.log_params(params)
        mlflow.log_metrics(m)
        fig, ax = plt.subplots(figsize=(4, 3.2))
        ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(name)
        mlflow.log_figure(fig, 'confusion_matrix.png')
        plt.close(fig)
        results.append({'模型': name, **{k: round(v, 4) for k, v in m.items()}})

pd.DataFrame(results).sort_values('f1', ascending=False)

,模型,accuracy,precision,recall,f1,roc_auc
2,RandomForest-200,0.8733,0.5875,0.9038,0.7121,0.9374
0,LogReg-C0.1,0.8433,0.5294,0.8654,0.6569,0.9418
1,LogReg-C1.0,0.8467,0.5366,0.8462,0.6567,0.9429


## 4. 去 MLflow UI 看对比

终端执行：
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 5001
```
打开 http://localhost:5001 ，能直接对比三组实验的参数与指标，点进单个 run 还能看到混淆矩阵图。

## 小结

- 不平衡数据先算基线，否则「83% 准确率」会让人误以为模型很好
- F1 与召回率是这类场景的核心指标：漏掉一个流失客户的成本远高于误报
- MLflow 的价值是**把每次尝试都留下证据**，而不是凭感觉说「调参后效果更好了」